# Iteration 1: replay check and the query budget (Jetson)

**Loss targeted:** the Jetson time budget. The submitted run lost 66 of 77 skips to the 22 s budget, not to matching.

**Hypothesis.** On the Jetson one query (render, maps, coarse FFT, DINO, LoFTR) costs about 2.2 s (13.06 s mean per photo, most photos run 6 queries).
The 22 s budget therefore holds about 10 queries. A photo that needs the second pass (18 queries) or today's escalation (27) cannot finish, so its
answer is lost. Two free or cheap levers: (a) agreement between the 6 standard queries needs **no extra query**; (b) the few extra queries that fit
should be the ones that recover most photos, in the best order, with early exit.

**Test (offline, from the rich dumps, no GPU).** Every photo has all 27 queries stored, so any policy is replayed exactly. Cost model: `S_PER_QUERY`
seconds per query on the Jetson (assumed, sensitivity shown), budget `BUDGET_S`. Settings (the extra queries) are chosen on official + NUDT 101 and
reported on NUDT 202 and the fresh NUDT 404.

Blocks: `train/checks/replay.py`, `train/checks/query_policy.py`.

In [ ]:
import os, sys, time
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)

from train.checks.replay import load_set, decide_current, score, paired_bootstrap, STANDARD, WIDE
from train.checks.query_policy import policy_table, greedy_extra_queries, run_policy

DUMPS = ROOT / "tests/Test_Iteration_6/results/rich_dump"
SET_NAMES = {"official": "official", "nudt101": "nudt_seed101", "nudt202": "nudt_seed202", "nudt404": "nudt_fresh404"}
CHOOSE_ON = ["official", "nudt101"]          # settings are chosen here
REPORT_ON = ["nudt202", "nudt404"]           # and only reported here
S_PER_QUERY = 2.2                            # Jetson seconds per query (assumption: 13.06 s mean / ~6 queries)
BUDGET_S = 22.0
sets = {}
for name, folder in SET_NAMES.items():
    if (DUMPS / folder).exists():
        sets[name] = load_set(DUMPS / folder)
print({k: len(v) for k, v in sets.items()})

## 1. Replay check: today's rule from the new dumps

Today's best rule (`agreement_query_pipeline.py`): 6 standard queries, answer at >= 8 inliers or >= 2 agreeing queries, else all 27 queries with
bar 11. The earlier measurements were official 379: 53.54 and NUDT 202 (200 unseen): 57.36, both 0 wrong. The new dumps should give about the same
(same code, same photos; only the GPU differs, so a few inlier counts may change).

In [ ]:
rows = []
current = {}
for name, photos in sets.items():
    t = pd.DataFrame([dict(photo=pid, **decide_current(p)) for pid, p in photos.items()])
    current[name] = t
    rows.append(dict(set=name, photos=len(t), score=round(score(t.points), 2), wrong=int((t.points < 0).sum()), answered=round(t.answered.mean(), 3),
                     **{"stage " + k: int(v) for k, v in t.stage.value_counts().items()}, mean_queries=round(t.n_queries.mean(), 2),
                     pc_s_per_photo=round(t.seconds.mean(), 2), photos_over_budget_jetson=int((t.n_queries * S_PER_QUERY > BUDGET_S).sum())))
display(pd.DataFrame(rows))
old = {"official": ROOT / "tests/Test_Iteration_3/results/fresh_run/agreement_pipeline_official.csv",
       "nudt202": ROOT / "tests/Test_Iteration_3/results/fresh_run/agreement_pipeline_nudt_seed202.csv"}
for name, path in old.items():
    if name in current and path.exists():
        o = pd.read_csv(path)
        from train.checks.replay import points as points_of
        o["old_points"] = [points_of(e) if a else 0 for a, e in zip(o.answered, o.error_sigma)]
        o["old_stage"] = o.stage
        m = current[name].merge(o[["photo", "old_points", "old_stage"]], on="photo")
        print(pd.crosstab(m.old_stage, m.stage))
        print("%s: %d photos in both, same points on %d; old score %.2f, new score %.2f on the same photos"
              % (name, len(m), int((m.points == m.old_points).sum()), score(m.old_points), score(m.points)))

## 2. What the Jetson does today, and today's rule under the budget

* **Jetson today** (submitted code): 6 standard queries at bar 8 (no agreement), then a second pass to 18 queries (9 headings x scales 1.0, 1.05),
  still bar 8, cut by the budget. Replayed here with M = 10 (the Jetson uses M = 5; see section 4).
* **Today's rule, capped**: agreement + escalation, but only as many queries as the budget holds.
* The cap is `floor(BUDGET_S / S_PER_QUERY)` queries; shown for 1.8 / 2.2 / 2.6 s per query.

In [ ]:
SECOND_PASS = [(h, s) for s in (1.0, 1.05) for h in (-10.0, -7.5, -5.0, -2.5, 0.0, 2.5, 5.0, 7.5, 10.0)]
def summarise(table, label, **extra):
    return dict(policy=label, score=round(score(table.points), 2), wrong=int((table.points < 0).sum()), answered=round(table.answered.mean(), 3),
                mean_q=round(table.n_queries.mean(), 2), p95_q=float(np.percentile(table.n_queries, 95)), max_q=int(table.n_queries.max()), **extra)
rows = []
for s_per_q in (1.8, 2.2, 2.6):
    cap = int(BUDGET_S // s_per_q)
    for name, photos in sets.items():
        jet = policy_table(photos, agree_min=99, escalation_bar=8, wide_order=SECOND_PASS, max_queries=cap)
        today_cap = policy_table(photos, max_queries=cap)
        today_free = policy_table(photos)
        bar_only = policy_table(photos, agree_min=99, max_queries=6)
        agree_only = policy_table(photos, max_queries=6)
        for t, label in ((bar_only, "bar 8 only, 6 queries"), (jet, "Jetson today (2nd pass, bar 8), capped"), (agree_only, "bar or agreement, 6 queries"),
                         (today_cap, "today's rule (agree + esc 11), capped"), (today_free, "today's rule, no cap")):
            rows.append(summarise(t, label, set=name, s_per_query=s_per_q, cap=cap))
budget = pd.DataFrame(rows)
display(budget[budget.s_per_query == S_PER_QUERY].pivot_table(index="policy", columns="set", values=["score", "wrong"], aggfunc="first"))
display(budget.pivot_table(index=["policy"], columns=["s_per_query", "set"], values="score", aggfunc="first"))

## 3. Which extra queries fit the budget best (chosen on official + NUDT 101)

Greedy: after the 6 standard queries and the agreement rule, add escalation queries one by one, each time the one that raises the summed points
over the choosing sets most (a wrong answer costs 1000 extra, so the choice keeps 0 wrong). Only photos not answered by the 6 queries can change,
so the search runs on those. Then the chosen order is replayed on every set with the cap the budget allows, and early exit inside the standard
queries is tried (answer as soon as the pooled choice has >= E inliers).

In [ ]:
cap = int(BUDGET_S // S_PER_QUERY)
hard = []
for name in CHOOSE_ON:
    if name in sets:
        hard.append({pid: p for pid, p in sets[name].items() if not run_policy(p, max_queries=6)["answered"]})
print("photos still skipped after 6 queries, choosing sets:", [len(h) for h in hard])
start = time.time()
order, trace = greedy_extra_queries(hard, k=21)
print("greedy order (%.0f s):" % (time.time() - start))
for i, (q, v) in enumerate(zip(order, trace)):
    print("  %2d  heading %+5.1f  scale %.2f   summed points over the choosing sets %d" % (i + 1, q[0], q[1], v))
outer_first = sorted([q for q in WIDE if q not in STANDARD], key=lambda q: (-abs(q[0]), abs(q[1] - 1.0)))

In [ ]:
rows = []
for name, photos in sets.items():
    base = policy_table(photos)
    for label, kw in (("today's rule, no cap", {}),
                      ("today's rule, capped", dict(max_queries=cap)),
                      ("greedy order, capped", dict(wide_order=order, max_queries=cap)),
                      ("outer headings first, capped", dict(wide_order=outer_first, max_queries=cap)),
                      ("greedy order, capped, early exit 15", dict(wide_order=order, max_queries=cap, early_bar=15)),
                      ("greedy order, capped, early exit 20", dict(wide_order=order, max_queries=cap, early_bar=20)),
                      ("greedy order, capped, early exit 30", dict(wide_order=order, max_queries=cap, early_bar=30))):
        t = policy_table(photos, **kw)
        gain = paired_bootstrap(budget_ref := policy_table(photos, agree_min=99, escalation_bar=8, wide_order=SECOND_PASS, max_queries=cap)["points"], t["points"])
        rows.append(summarise(t, label, set=name, vs_jetson_today="%+.2f [%+.2f, %+.2f]" % gain))
res = pd.DataFrame(rows)
display(res.pivot_table(index="policy", columns="set", values=["score", "wrong", "mean_q"], aggfunc="first"))
display(res[["set", "policy", "vs_jetson_today"]].pivot(index="policy", columns="set", values="vs_jetson_today"))

## 4. M = 5 (the Jetson's setting) against M = 10

The Jetson runs only the first 5 DINO-ordered candidates of each query through LoFTR. The fine stage is about half of a query's time on the PC, so
M = 5 saves about a quarter of it. Replayed by cutting every query to its first 5 candidates.

In [ ]:
from train.checks.replay import truncate_m
rows = []
for name, photos in sets.items():
    p5 = {pid: truncate_m(p, 5) for pid, p in photos.items()}
    for label, ph in (("M=10", photos), ("M=5", p5)):
        for pl, kw in (("today's rule, no cap", {}), ("greedy order, capped", dict(wide_order=order, max_queries=cap))):
            rows.append(summarise(policy_table(ph, **kw), pl + ", " + label, set=name))
display(pd.DataFrame(rows).pivot_table(index="policy", columns="set", values=["score", "wrong"], aggfunc="first"))

In [ ]:
out = ROOT / "tests/Test_Iteration_6/results"
budget.to_csv(out / "it1_budget_policies.csv", index=False); res.to_csv(out / "it1_greedy_policies.csv", index=False)
pd.DataFrame(dict(heading=[q[0] for q in order], scale=[q[1] for q in order], summed_points=trace)).to_csv(out / "it1_greedy_order.csv", index=False)
print("saved")